In [19]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from imblearn.over_sampling import SMOTE
import plotly.graph_objects as go
import warnings

warnings.filterwarnings('ignore')

# Verificar se GPU está disponível
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando: {device}")
print(f"PyTorch versão: {torch.__version__}")

Usando: cpu
PyTorch versão: 2.11.0+cpu


## Deep Learning com PyTorch

**Por que usar Deep Learning aqui se já temos ML clássico?**
Para dados tabulares (tabelas), o ML clássico frequentemente ganha. 
Para imagens, áudio e texto, o Deep Learning domina. 
Comparar as duas abordagens é uma decisão técnica madura.

**O que é uma rede neural?**
É um conjunto de camadas de "neurônios" matemáticos conectados.
Cada camada transforma os dados, extraindo padrões cada vez mais complexos.
A última camada dá a previsão final.

In [20]:
# Carregar e preparar dados (mesmo processo do notebook anterior)
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df = df.drop(['customerID', 'gender', 'PhoneService'], axis=1)
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

categoricas = df.select_dtypes(include='object').columns.tolist()
df_ml = pd.get_dummies(df, columns=categoricas, drop_first=True)

X = df_ml.drop('Churn', axis=1)
y = df_ml['Churn']

# Divisão treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# SMOTE
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

# Normalizar
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

print(f"Treino: {X_train_scaled.shape}")
print(f"Teste:  {X_test_scaled.shape}")

Treino: (8278, 28)
Teste:  (1409, 28)


### O que são Tensors?
Tensors são a estrutura de dados fundamental do PyTorch — é como um 
array NumPy, mas com superpoderes: pode rodar na GPU e o PyTorch 
calcula automaticamente os gradientes (derivadas) pra treinar a rede.

In [21]:
# Converter para Tensors do PyTorch
X_train_tensor = torch.FloatTensor(X_train_scaled)
y_train_tensor = torch.FloatTensor(y_train_bal.values)
X_test_tensor = torch.FloatTensor(X_test_scaled)
y_test_tensor = torch.FloatTensor(y_test.values)

# Criar DataLoaders (divide os dados em batches pra treinar)
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

print(f"Total de batches por epoch: {len(train_loader)}")
print(f"Tamanho de cada batch: 64")
print(f"Input size: {X_train_tensor.shape[1]} features")

Total de batches por epoch: 130
Tamanho de cada batch: 64
Input size: 28 features


### Arquitetura da Rede Neural

A rede vai ter 4 camadas:
- **Input:** 28 features (uma por variável)
- **Hidden 1:** 128 neurônios + ReLU + Dropout
- **Hidden 2:** 64 neurônios + ReLU + Dropout  
- **Hidden 3:** 32 neurônios + ReLU
- **Output:** 1 neurônio + Sigmoid (dá probabilidade entre 0 e 1)

**ReLU** é a função de ativação — ela introduz não-linearidade, 
permitindo que a rede aprenda padrões complexos.

**Dropout** desliga aleatoriamente alguns neurônios durante o treino, 
forçando a rede a não depender demais de nenhum neurônio específico.
Isso evita overfitting (decorar os dados de treino).

**Sigmoid** na última camada transforma o output em probabilidade (0 a 1).

In [22]:
# Definir a arquitetura da rede neural
class ChurnNet(nn.Module):
    def __init__(self, input_size):
        super(ChurnNet, self).__init__()
        
        self.network = nn.Sequential(
            # Camada 1
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            # Camada 2
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            # Camada 3
            nn.Linear(64, 32),
            nn.ReLU(),
            
            # Output
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.network(x)

# Instanciar o modelo
model = ChurnNet(input_size=28).to(device)

# Contar parâmetros
total_params = sum(p.numel() for p in model.parameters())
print(f"Arquitetura da rede:")
print(model)
print(f"\nTotal de parâmetros: {total_params:,}")

Arquitetura da rede:
ChurnNet(
  (network): Sequential(
    (0): Linear(in_features=28, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): ReLU()
    (8): Linear(in_features=32, out_features=1, bias=True)
    (9): Sigmoid()
  )
)

Total de parâmetros: 14,081


### Treinamento da Rede Neural

**Loss function (BCELoss):** Mede o erro da rede em cada batch.
Binary Cross Entropy é a função padrão pra classificação binária.

**Optimizer (Adam):** Ajusta os pesos da rede pra minimizar o erro.
Adam é o optimizer mais usado — adapta a taxa de aprendizado automaticamente.

**Epoch:** Uma passagem completa por todos os dados de treino.
Vamos treinar por 50 epochs — a rede vai melhorar progressivamente.

In [23]:
# Configurar treino
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

# Listas pra guardar o histórico
train_losses = []
val_losses = []
train_accs = []
val_accs = []

# Loop de treinamento
n_epochs = 50

for epoch in range(n_epochs):
    # === TREINO ===
    model.train()
    batch_losses = []
    batch_accs = []
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        y_pred = model(X_batch).squeeze()
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        
        batch_losses.append(loss.item())
        acc = ((y_pred > 0.5).float() == y_batch).float().mean()
        batch_accs.append(acc.item())
    
    # === VALIDAÇÃO ===
    model.eval()
    with torch.no_grad():
        y_val_pred = model(X_test_tensor.to(device)).squeeze()
        val_loss = criterion(y_val_pred, y_test_tensor.to(device))
        val_acc = ((y_val_pred > 0.5).float() == y_test_tensor.to(device)).float().mean()
    
    train_loss = np.mean(batch_losses)
    train_acc = np.mean(batch_accs)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss.item())
    train_accs.append(train_acc)
    val_accs.append(val_acc.item())
    
    scheduler.step(val_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{n_epochs} | "
              f"Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
              f"Acc: {train_acc*100:.1f}% | Val Acc: {val_acc*100:.1f}%")

Epoch  10/50 | Loss: 0.3633 | Val Loss: 0.4544 | Acc: 83.5% | Val Acc: 78.1%
Epoch  20/50 | Loss: 0.3378 | Val Loss: 0.4543 | Acc: 84.9% | Val Acc: 78.0%
Epoch  30/50 | Loss: 0.3141 | Val Loss: 0.4507 | Acc: 85.8% | Val Acc: 77.9%
Epoch  40/50 | Loss: 0.3068 | Val Loss: 0.4513 | Acc: 85.8% | Val Acc: 78.0%
Epoch  50/50 | Loss: 0.3061 | Val Loss: 0.4518 | Acc: 86.4% | Val Acc: 77.7%


In [24]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Visualizar histórico de treino
fig = make_subplots(rows=1, cols=2, subplot_titles=('Loss por Epoch', 'Acurácia por Epoch'))

fig.add_trace(go.Scatter(y=train_losses, name='Treino', line=dict(color='#3B82F6', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(y=val_losses, name='Validação', line=dict(color='#EF4444', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(y=train_accs, name='Treino', line=dict(color='#3B82F6', width=2), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(y=val_accs, name='Validação', line=dict(color='#EF4444', width=2), showlegend=False), row=1, col=2)

fig.update_layout(
    title=dict(text='Histórico de Treinamento — Rede Neural', font=dict(size=20)),
    template='plotly_white', height=400
)
fig.show()

# Avaliação final
model.eval()
with torch.no_grad():
    y_proba_nn = model(X_test_tensor.to(device)).squeeze().cpu().numpy()
    y_pred_nn = (y_proba_nn > 0.5).astype(int)

print("\nRESULTADOS — REDE NEURAL")
print(f"{'='*50}")
print(f"Acurácia:  {accuracy_score(y_test, y_pred_nn)*100:.1f}%")
print(f"AUC-ROC:   {roc_auc_score(y_test, y_proba_nn):.3f}")
print(f"\n{classification_report(y_test, y_pred_nn, target_names=['Fica', 'Cancela'])}")


RESULTADOS — REDE NEURAL
Acurácia:  77.7%
AUC-ROC:   0.831

              precision    recall  f1-score   support

        Fica       0.85      0.85      0.85      1035
     Cancela       0.58      0.57      0.58       374

    accuracy                           0.78      1409
   macro avg       0.71      0.71      0.71      1409
weighted avg       0.78      0.78      0.78      1409



### Comparação Final — ML Clássico vs Deep Learning

| Modelo | Acurácia | AUC-ROC |
|--------|----------|---------|
| Logistic Regression | 74.8% | 0.808 |
| Random Forest | 76.7% | 0.836 |
| Gradient Boosting (otimizado) | 77.7% | 0.819 |
| **Rede Neural (PyTorch)** | **77.5%** | **0.828** |

**Conclusão técnica:**
A rede neural empata com o Gradient Boosting otimizado, mas com muito 
mais complexidade. Para dados tabulares, ML clássico é tão bom quanto 
Deep Learning — e mais interpretável. A rede neural brilha em imagens, 
texto e áudio, onde os padrões são muito mais complexos.

**Conclusão de negócio:**
Adotamos o Gradient Boosting como modelo de produção por ser levemente 
superior, mais rápido e mais interpretável via SHAP.

In [25]:
# Salvar o modelo PyTorch
torch.save(model.state_dict(), '../models/churn_neural_net.pt')
print("Modelo PyTorch salvo: models/churn_neural_net.pt")

Modelo PyTorch salvo: models/churn_neural_net.pt
